In [ ]:
import os, sys, subprocess, time, requests
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/hacker-society'):
    subprocess.run(['git', 'clone', 'https://github.com/blah-blah-cell/hacker-society.git', '/kaggle/working/hacker-society'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/hacker-society', 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/hacker-society', 'reset', '--hard', 'origin/main'], check=True)
os.chdir('/kaggle/working/hacker-society')
subprocess.run(['pip', 'install', '-q', 'fastapi', 'uvicorn', 'transformers', 'accelerate', 'torch'], check=True)
subprocess.run(['apt-get', 'update', '-q'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'nmap', 'mysql-client', 'nano', 'vsftpd', 'net-tools', 'ufw', 'curl', 'netcat', 'psmisc'], check=True)
subprocess.run(['service', 'vsftpd', 'start'], capture_output=True)
subprocess.run('pkill -9 -f vllm ; pkill -9 -f python ; fuser -v /dev/nvidia* -k -9', shell=True, capture_output=True)
time.sleep(4)
print('=== Starting Qwen 2.5 1.5B Server (Qwen/Qwen2.5-1.5B-Instruct) ===')
server_log = open('/tmp/server.log', 'w')
server_proc = subprocess.Popen([sys.executable, 'fast_server.py'], stdout=server_log, stderr=subprocess.STDOUT)
print('Waiting up to 120s for server startup...')
ready = False
for i in range(120):
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=2)
        if r.status_code == 200:
            print(f'=== SERVER IS UP AND READY (after {i*2}s) ===')
            print(r.json())
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)
if not ready:
    print('Server failed to start within 120s. Log:')
    with open('/tmp/server.log') as f:
        server_err = f.read()
        print(server_err)
        with open('/kaggle/working/match_output.txt', 'w') as out_f:
            out_f.write('SERVER_STARTUP_FAILED:\n' + server_err)
else:
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.free', '--format=csv'])
    print('=== Running Autonomous Cyber Match ===')
    res = subprocess.run(
        "echo '1' | MOCK_DOCKER_NO_CONTAINERS=1 REAL_LOCAL_SHELL=1 python -m src.main --model Qwen/Qwen2.5-1.5B-Instruct --base-url http://localhost:8000/v1 --attackers 1 --defenders 1 --turns 3",
        shell=True, capture_output=True, text=True, cwd='/kaggle/working/hacker-society'
    )
    match_out = res.stdout + ('\n=== STDERR ===\n' + res.stderr if res.stderr else '')
    with open('/kaggle/working/match_output.txt', 'w', encoding='utf-8') as f:
        f.write(match_out)
    print(match_out)


In [ ]:
import os, sys, subprocess, time, requests
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/hacker-society'):
    subprocess.run(['git', 'clone', 'https://github.com/blah-blah-cell/hacker-society.git', '/kaggle/working/hacker-society'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/hacker-society', 'pull', 'origin', 'main'], check=True)
os.chdir('/kaggle/working/hacker-society')
subprocess.run(['apt-get', 'update', '-q'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'nmap', 'mysql-client', 'nano', 'vsftpd', 'net-tools', 'ufw', 'curl', 'netcat'], check=True)
subprocess.run(['service', 'vsftpd', 'start'], capture_output=True)
with open('/tmp/flag.txt', 'w') as f:
    f.write('HACKSOC_REAL_FLAG_7B_2026')
subprocess.run(['pkill', '-9', '-f', 'vllm'], capture_output=True)
time.sleep(3)
print('=== Starting Qwen/Qwen2.5-7B-Instruct-AWQ ===')
vllm_log = open('/tmp/vllm7b.log', 'w')
vllm_proc = subprocess.Popen(
    [sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
     '--model', 'Qwen/Qwen2.5-7B-Instruct-AWQ',
     '--port', '8000',
     '--gpu-memory-utilization', '0.85',
     '--max-model-len', '4096',
     '--enable-auto-tool-choice',
     '--tool-call-parser', 'hermes'],
    stdout=vllm_log, stderr=subprocess.STDOUT
)
print('Waiting for vLLM server...')
ready = False
for i in range(90):
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=2)
        if r.status_code == 200:
            print(f'=== vLLM SERVER UP (after {i*2}s) ===')
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)
if not ready:
    print('Server failed. Log:')
    with open('/tmp/vllm7b.log') as f:
        print(f.read()[-2000:])
else:
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.free', '--format=csv'])
    print('=== Running 3-Turn Match (REAL_LOCAL_SHELL) ===')
    res = subprocess.run(
        "echo '1' | MOCK_DOCKER_NO_CONTAINERS=1 REAL_LOCAL_SHELL=1 python -m src.main --model Qwen/Qwen2.5-7B-Instruct-AWQ --base-url http://localhost:8000/v1 --attackers 1 --defenders 1 --turns 3",
        shell=True, capture_output=True, text=True, cwd='/kaggle/working/hacker-society'
    )
    print('=== STDOUT ===')
    print(res.stdout)
    if res.stderr:
        print('=== STDERR (tail) ===')
        print(res.stderr[-800:])
